# 🧹 Step 1 — Data Cleaning
**Laptop Price Prediction project**

In this notebook we take the raw dataset and turn it into a clean table that a machine-learning model can understand.

**What we will do:**
1. Load the data and look at it
2. Remove the useless column
3. Fill the missing values
4. Convert text columns (like `8GB`, `1.5kg`) into numbers
5. Create new useful columns (Touchscreen, IPS, PPI, HDD, SSD ...)
6. Save the clean data to `cleaned_laptop_data.csv` (used by the next two notebooks)

## 1. Import Libraries

In [1]:
import pandas as pd      # for working with tables
import numpy as np       # for maths
import re                # for finding numbers inside text

## 2. Load the Dataset

In [2]:
df = pd.read_csv("laptop_dataset_50000_ml.csv")

## 3. First Look at the Data

In [3]:
# (rows, columns)
df.shape

(50000, 12)

In [4]:
# first 5 rows
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Lenovo,Notebook,15.6,NaN,Intel Core i7 6500U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.77kg,42386.39
1,1,Toshiba,Notebook,13.3,Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 7,1.16kg,63392.07
2,2,HP,Notebook,17.3,Full HD 1920x1080,Intel Core i7 8550U 1.8GHz,8GB,256GB SSD,Nvidia GeForce 930MX,Windows 10,2.5kg,55677.60
3,3,Samsung,Ultrabook,15.0,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.23kg,91282.48
4,4,Asus,Gaming,17.1,IPS Panel Full HD 1920x1080,Intel Core i7 6820HK 2.7GHz,6GB,512GB SSD,Nvidia GeForce GTX 1080,Windows 10,3.8kg,153715.23


In [5]:
# column names, data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        50000 non-null  int64  
 1   Company           48724 non-null  str    
 2   TypeName          48722 non-null  str    
 3   Inches            48781 non-null  float64
 4   ScreenResolution  48747 non-null  str    
 5   Cpu               48712 non-null  str    
 6   Ram               48756 non-null  str    
 7   Memory            48760 non-null  str    
 8   Gpu               48779 non-null  str    
 9   OpSys             48731 non-null  str    
 10  Weight            48788 non-null  str    
 11  Price             50000 non-null  float64
dtypes: float64(2), int64(1), str(9)
memory usage: 9.7 MB


In [6]:
# are there any duplicate rows?
df.duplicated().sum()

np.int64(0)

In [7]:
# how many missing values in each column?
df.isnull().sum()

Unnamed: 0             0
Company             1276
TypeName            1278
Inches              1219
ScreenResolution    1253
Cpu                 1288
Ram                 1244
Memory              1240
Gpu                 1221
OpSys               1269
Weight              1212
Price                  0
dtype: int64

**What we see:** every column (except `Price`) has around 1,200 missing values, and columns like `Ram` and `Weight` are stored as text (`8GB`, `1.5kg`). We will fix both.

## 4. Remove the Unused Column
`Unnamed: 0` is just the old row number. It does not help in predicting price.

In [8]:
df.drop(columns=["Unnamed: 0"], inplace=True)
df.head()

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,Lenovo,Notebook,15.6,NaN,Intel Core i7 6500U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.77kg,42386.39
1,Toshiba,Notebook,13.3,Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 7,1.16kg,63392.07
2,HP,Notebook,17.3,Full HD 1920x1080,Intel Core i7 8550U 1.8GHz,8GB,256GB SSD,Nvidia GeForce 930MX,Windows 10,2.5kg,55677.60
3,Samsung,Ultrabook,15.0,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.23kg,91282.48
4,Asus,Gaming,17.1,IPS Panel Full HD 1920x1080,Intel Core i7 6820HK 2.7GHz,6GB,512GB SSD,Nvidia GeForce GTX 1080,Windows 10,3.8kg,153715.23


## 5. Handle Missing Values
* **Number columns** → fill with the **median** (the middle value)
* **Text columns** → fill with the **mode** (the most common value)

In [9]:
numeric_cols = df.select_dtypes(include="number").columns
text_cols = df.select_dtypes(exclude="number").columns

print("Numeric columns:", list(numeric_cols))
print("Text columns   :", list(text_cols))

Numeric columns: ['Inches', 'Price']
Text columns   : ['Company', 'TypeName', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight']


In [10]:
# fill numeric columns with median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [11]:
# fill text columns with mode
for col in text_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [12]:
# check again - everything should be 0 now
df.isnull().sum()

Company             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Memory              0
Gpu                 0
OpSys               0
Weight              0
Price               0
dtype: int64

## 6. Clean `Ram` and `Weight`
Remove the words `GB` and `kg`, then convert to numbers.

In [13]:
df["Ram"] = df["Ram"].str.replace("GB", "").astype(int)
df["Weight"] = df["Weight"].str.replace("kg", "").astype(float)
df[["Ram", "Weight"]].head()

,Ram,Weight
0,4,1.77
1,4,1.16
2,8,2.50
3,8,1.23
4,6,3.80


## 7. Screen Resolution → 3 useful features
The column looks like `IPS Panel Full HD / Touchscreen 1920x1080`. It hides three facts:
* is it a **Touchscreen**?
* is it an **IPS** panel?
* what is the pixel **width × height**?

In [14]:
df["ScreenResolution"].value_counts().head(10)

ScreenResolution
Full HD 1920x1080                                20248
1366x768                                         10488
IPS Panel Full HD 1920x1080                       8450
IPS Panel Full HD / Touchscreen 1920x1080         2057
Full HD / Touchscreen 1920x1080                   1725
1600x900                                           824
Touchscreen 1366x768                               634
Quad HD+ / Touchscreen 3200x1800                   520
IPS Panel 4K Ultra HD 3840x2160                    489
IPS Panel 4K Ultra HD / Touchscreen 3840x2160      432
Name: count, dtype: int64

In [15]:
# 1 if the word is present, else 0
df["Touchscreen"] = df["ScreenResolution"].apply(lambda x: 1 if "Touchscreen" in x else 0)
df["Ips"] = df["ScreenResolution"].apply(lambda x: 1 if "IPS" in x else 0)

In [16]:
# pull out the width and height  (example: 1920x1080 -> 1920 and 1080)
res = df["ScreenResolution"].str.extract(r"(\d+)x(\d+)")
df["X_res"] = res[0].astype(int)
df["Y_res"] = res[1].astype(int)
df[["ScreenResolution", "X_res", "Y_res"]].head()

,ScreenResolution,X_res,Y_res
0,Full HD 1920x1080,1920,1080
1,Full HD 1920x1080,1920,1080
2,Full HD 1920x1080,1920,1080
3,Full HD 1920x1080,1920,1080
4,IPS Panel Full HD 1920x1080,1920,1080


### Pixels Per Inch (PPI)
PPI tells how sharp the screen is:  
`PPI = √(width² + height²) / screen size in inches`

In [17]:
df["ppi"] = ((df["X_res"] ** 2 + df["Y_res"] ** 2) ** 0.5) / df["Inches"]
df[["X_res", "Y_res", "Inches", "ppi"]].head()

,X_res,Y_res,Inches,ppi
0,1920,1080,15.6,141.211998
1,1920,1080,13.3,165.632118
2,1920,1080,17.3,127.335675
3,1920,1080,15.0,146.860478
4,1920,1080,17.1,128.824981


In [18]:
# these columns are no longer needed
df.drop(columns=["ScreenResolution", "X_res", "Y_res", "Inches"], inplace=True)
df.head()

,Company,TypeName,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price,Touchscreen,Ips,ppi
0,Lenovo,Notebook,Intel Core i7 6500U 2.5GHz,4,256GB SSD,Intel HD Graphics 520,Windows 10,1.77,42386.39,0,0,141.211998
1,Toshiba,Notebook,Intel Core i5 6200U 2.3GHz,4,128GB SSD,Intel HD Graphics 520,Windows 7,1.16,63392.07,0,0,165.632118
2,HP,Notebook,Intel Core i7 8550U 1.8GHz,8,256GB SSD,Nvidia GeForce 930MX,Windows 10,2.50,55677.60,0,0,127.335675
3,Samsung,Ultrabook,Intel Core i7 7500U 2.7GHz,8,256GB SSD,Intel HD Graphics 620,Windows 10,1.23,91282.48,0,0,146.860478
4,Asus,Gaming,Intel Core i7 6820HK 2.7GHz,6,512GB SSD,Nvidia GeForce GTX 1080,Windows 10,3.80,153715.23,0,1,128.824981


## 8. CPU → CPU Brand
The CPU text is very long (`Intel Core i7 6500U 2.5GHz`). We shrink it into 5 simple groups.

In [19]:
df["Cpu"].value_counts().head(10)

Cpu
Intel Core i5 7200U 2.5GHz     8362
Intel Core i7 7700HQ 2.8GHz    5512
Intel Core i7 7500U 2.7GHz     4938
Intel Core i7 8550U 1.8GHz     2807
Intel Core i5 8250U 1.6GHz     2639
Intel Core i5 6200U 2.3GHz     2599
Intel Core i3 6006U 2GHz       2427
Intel Core i7 6500U 2.5GHz     1859
Intel Core i7 6700HQ 2.6GHz    1620
Intel Core i3 7100U 2.4GHz     1392
Name: count, dtype: int64

In [20]:
def get_cpu_brand(text):
    # keep only the first three words, e.g. "Intel Core i7"
    name = " ".join(text.split()[0:3])

    if name in ["Intel Core i7", "Intel Core i5", "Intel Core i3"]:
        return name
    elif name.split()[0] == "Intel":
        return "Other Intel Processor"
    else:
        return "AMD Processor"

df["Cpu brand"] = df["Cpu"].apply(get_cpu_brand)
df["Cpu brand"].value_counts()

Cpu brand
Intel Core i7            19787
Intel Core i5            17051
Other Intel Processor     5740
Intel Core i3             5071
AMD Processor             2351
Name: count, dtype: int64

In [21]:
df.drop(columns=["Cpu"], inplace=True)

## 9. Memory → HDD and SSD
Memory looks like `128GB SSD +  1TB HDD`. We split it into two number columns: **HDD** and **SSD** (in GB).  
*(1 TB is counted as 1000 GB.)*

In [22]:
df["Memory"].value_counts().head(10)

Memory
256GB SSD               16489
1TB HDD                  8303
500GB HDD                4955
512GB SSD                4406
128GB SSD +  1TB HDD     3497
128GB SSD                2953
256GB SSD +  1TB HDD     2737
32GB Flash Storage       1435
2TB HDD                   614
512GB SSD +  1TB HDD      564
Name: count, dtype: int64

In [23]:
def split_memory(text):
    hdd = 0
    ssd = 0

    # a laptop can have two drives separated by "+"
    for part in text.split("+"):
        part = part.strip()
        size = float(re.findall(r"[\d.]+", part)[0])   # the number, e.g. 128 or 1.0

        if "TB" in part:
            size = size * 1000                          # TB -> GB

        if "HDD" in part:
            hdd = hdd + size
        elif "SSD" in part:
            ssd = ssd + size
        # Hybrid and Flash Storage are ignored (very rare)

    return pd.Series([hdd, ssd])

df[["HDD", "SSD"]] = df["Memory"].apply(split_memory)
df[["Memory", "HDD", "SSD"]].head(8)

,Memory,HDD,SSD
0,256GB SSD,0.0,256.0
1,128GB SSD,0.0,128.0
2,256GB SSD,0.0,256.0
3,256GB SSD,0.0,256.0
4,512GB SSD,0.0,512.0
5,256GB SSD,0.0,256.0
6,256GB SSD + 1TB HDD,1000.0,256.0
7,256GB SSD + 1TB HDD,1000.0,256.0


In [24]:
df["HDD"] = df["HDD"].astype(int)
df["SSD"] = df["SSD"].astype(int)
df.drop(columns=["Memory"], inplace=True)

## 10. GPU → GPU Brand
We keep only the brand name (the first word).

In [25]:
df["Gpu brand"] = df["Gpu"].apply(lambda x: x.split()[0])
df["Gpu brand"].value_counts()

Gpu brand
Intel     28145
Nvidia    15054
AMD        6768
ARM          33
Name: count, dtype: int64

In [26]:
# 'ARM' appears only a few times, so we remove those rows
df = df[df["Gpu brand"] != "ARM"]
df.drop(columns=["Gpu"], inplace=True)

## 11. Operating System → 3 groups

In [27]:
df["OpSys"].value_counts()

OpSys
Windows 10      41444
No OS            2414
Linux            2260
Windows 7        1743
Chrome OS         959
macOS             481
Mac OS X          306
Windows 10 S      275
Android            85
Name: count, dtype: int64

In [28]:
def get_os(name):
    if name in ["Windows 10", "Windows 7", "Windows 10 S"]:
        return "Windows"
    elif name in ["macOS", "Mac OS X"]:
        return "Mac"
    else:
        return "Others/No OS/Linux"

df["os"] = df["OpSys"].apply(get_os)
df.drop(columns=["OpSys"], inplace=True)
df["os"].value_counts()

os
Windows               43462
Others/No OS/Linux     5718
Mac                     787
Name: count, dtype: int64

## 12. Final Check

In [29]:
# keep the columns in a fixed, tidy order
df = df[["Company", "TypeName", "Ram", "Weight", "Touchscreen", "Ips", "ppi",
         "Cpu brand", "HDD", "SSD", "Gpu brand", "os", "Price"]]
df = df.reset_index(drop=True)
df.head()

,Company,TypeName,Ram,Weight,Touchscreen,Ips,ppi,Cpu brand,HDD,SSD,Gpu brand,os,Price
0,Lenovo,Notebook,4,1.77,0,0,141.211998,Intel Core i7,0,256,Intel,Windows,42386.39
1,Toshiba,Notebook,4,1.16,0,0,165.632118,Intel Core i5,0,128,Intel,Windows,63392.07
2,HP,Notebook,8,2.50,0,0,127.335675,Intel Core i7,0,256,Nvidia,Windows,55677.60
3,Samsung,Ultrabook,8,1.23,0,0,146.860478,Intel Core i7,0,256,Intel,Windows,91282.48
4,Asus,Gaming,6,3.80,0,1,128.824981,Intel Core i7,0,512,Nvidia,Windows,153715.23


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49967 entries, 0 to 49966
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Company      49967 non-null  str    
 1   TypeName     49967 non-null  str    
 2   Ram          49967 non-null  int64  
 3   Weight       49967 non-null  float64
 4   Touchscreen  49967 non-null  int64  
 5   Ips          49967 non-null  int64  
 6   ppi          49967 non-null  float64
 7   Cpu brand    49967 non-null  str    
 8   HDD          49967 non-null  int64  
 9   SSD          49967 non-null  int64  
 10  Gpu brand    49967 non-null  str    
 11  os           49967 non-null  str    
 12  Price        49967 non-null  float64
dtypes: float64(3), int64(5), str(5)
memory usage: 6.9 MB


In [31]:
print("Missing values:", df.isnull().sum().sum())
print("Shape:", df.shape)

Missing values: 0
Shape: (49967, 13)


In [32]:
df.describe()

,Ram,Weight,Touchscreen,Ips,ppi,HDD,SSD,Price
count,49967.000000,49967.000000,49967.000000,49967.000000,49967.000000,49967.000000,49967.000000,49967.000000
mean,9.310665,2.044838,0.144415,0.272820,146.621029,405.531971,185.959934,60146.073227
std,7.855070,0.660080,0.351514,0.445414,42.734002,515.232064,185.915274,37387.915929
min,2.000000,0.690000,0.000000,0.000000,85.168090,0.000000,0.000000,8284.920000
25%,4.000000,1.530000,0.000000,0.000000,126.603860,0.000000,0.000000,32261.560000
50%,8.000000,2.060000,0.000000,0.000000,141.211998,0.000000,256.000000,52021.710000
75%,8.000000,2.310000,0.000000,1.000000,157.350512,1000.000000,256.000000,79200.900000
max,64.000000,4.780000,1.000000,1.000000,383.114290,2000.000000,1024.000000,345223.120000


## 13. Save the Clean Data
The next notebooks (Visualization and ML Model) will load this file.

In [33]:
df.to_csv("cleaned_laptop_data.csv", index=False)
print("✅ cleaned_laptop_data.csv saved!")

✅ cleaned_laptop_data.csv saved!
